# Полная оценка всех моделей на GQA-ru и MMBench-ru

Ноутбук последовательно оценивает шесть вариантов моделей на всех доступных тестовых данных:

- Qwen2.5-VL baseline;
- Qwen2.5-VL + LoRA #1;
- Qwen2.5-VL + LoRA #2;
- Qwen2.5-VL + LoRA #3;
- SmolVLM baseline;
- SmolVLM + LoRA.

Используются все **12 216** вопросов `testdev` GQA-ru и все **3 910** примеров `dev` MMBench-ru. После полного успешного запуска ноутбук перезаписывает CSV в `reports/evaluation` в формате, который уже читает `evaluate_all_models.ipynb`.

Долгий инференс защищён промежуточными checkpoint-файлами. При повторном запуске готовые строки не вычисляются заново. Итоговые CSV публикуются только после проверки полного покрытия; прежние файлы предварительно копируются в каталог резервной копии.

## 1. Импорты

In [1]:
from pathlib import Path
from io import BytesIO
from datetime import datetime
import gc
import hashlib
import json
import re
import shutil
import time

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
import torch
from tqdm.auto import tqdm

from peft import PeftModel
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, set_seed

try:
    from transformers import AutoModelForImageTextToText as SmolVLMModelClass
except ImportError:
    try:
        from transformers import AutoModelForVision2Seq as SmolVLMModelClass
    except ImportError:
        from transformers import Idefics3ForConditionalGeneration as SmolVLMModelClass

from qwen_vl_utils import process_vision_info

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Конфигурация полного запуска

In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень проекта")


ROOT = find_project_root()
GQA_DIR = ROOT / "data" / "GQA-ru"
GQA_INSTRUCTIONS_PATH = GQA_DIR / "testdev_balanced_instructions" / "testdev-00000-of-00001.parquet"
GQA_IMAGES_PATH = GQA_DIR / "testdev_balanced_images" / "testdev-00000-of-00001.parquet"
MMBENCH_PATH = ROOT / "data" / "MMBench-ru" / "mmbench_ru_dev.parquet"

QWEN_MODEL_DIR = ROOT / "models" / "Qwen2.5-VL-3B-Instruct"
SMOL_MODEL_DIR = ROOT / "models" / "SmolVLM-Instruct"
REPORT_DIR = ROOT / "reports" / "evaluation"

# None означает полный датасет. При отладке можно поставить небольшое число,
# но публикация основных CSV в таком режиме будет заблокирована.
MAX_GQA_SAMPLES = None
MAX_MMBENCH_SAMPLES = None
SEED = 42
MAX_NEW_TOKENS = 8
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28
SMOL_LONGEST_EDGE = 768
CHECKPOINT_EVERY = 250
REQUIRE_CUDA = True
STRICT_FULL_COVERAGE = True

run_suffix = f"gqa_{MAX_GQA_SAMPLES or 'all'}__mmbench_{MAX_MMBENCH_SAMPLES or 'all'}"
CHECKPOINT_DIR = REPORT_DIR / "full_eval_checkpoints" / run_suffix
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    {
        "slug": "qwen_base",
        "name": "Qwen2.5-VL базовая",
        "family": "qwen",
        "model_dir": QWEN_MODEL_DIR,
        "adapter_dir": None,
    },
    {
        "slug": "qwen_lora_1",
        "name": "Qwen2.5-VL + LoRA #1",
        "family": "qwen",
        "model_dir": QWEN_MODEL_DIR,
        "adapter_dir": ROOT / "outputs" / "qwen_lora_adapter",
    },
    {
        "slug": "qwen_lora_2",
        "name": "Qwen2.5-VL + LoRA #2",
        "family": "qwen",
        "model_dir": QWEN_MODEL_DIR,
        "adapter_dir": ROOT / "outputs" / "qwen_lora_adapter_2",
    },
    {
        "slug": "qwen_lora_3",
        "name": "Qwen2.5-VL + LoRA #3",
        "family": "qwen",
        "model_dir": QWEN_MODEL_DIR,
        "adapter_dir": ROOT / "outputs" / "qwen_lora_adapter_3",
    },
    {
        "slug": "smol_base",
        "name": "SmolVLM базовая",
        "family": "smolvlm",
        "model_dir": SMOL_MODEL_DIR,
        "adapter_dir": None,
    },
    {
        "slug": "smol_lora",
        "name": "SmolVLM + LoRA",
        "family": "smolvlm",
        "model_dir": SMOL_MODEL_DIR,
        "adapter_dir": ROOT / "outputs" / "smolvlm_lora_adapter",
    },
]

GQA_EXPORTS = {
    "qwen_lora_1": ("qwen_lora_predictions.csv", "qwen_lora_metrics.csv", "qwen_base"),
    "qwen_lora_2": ("qwen_lora_v2_predictions.csv", "qwen_lora_v2_metrics.csv", "qwen_base"),
    "qwen_lora_3": ("qwen_lora_v3_predictions.csv", "qwen_lora_v3_metrics.csv", "qwen_base"),
    "smol_lora": ("smolvlm_lora_predictions.csv", "smolvlm_lora_metrics.csv", "smol_base"),
}

set_seed(SEED)
MODEL_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

pd.DataFrame([
    {"Параметр": "Корень проекта", "Значение": str(ROOT)},
    {"Параметр": "GQA-ru", "Значение": str(GQA_INSTRUCTIONS_PATH)},
    {"Параметр": "MMBench-ru", "Значение": str(MMBENCH_PATH)},
    {"Параметр": "CUDA", "Значение": torch.cuda.is_available()},
    {"Параметр": "GPU", "Значение": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет"},
    {"Параметр": "dtype", "Значение": str(MODEL_DTYPE)},
    {"Параметр": "Checkpoint", "Значение": str(CHECKPOINT_DIR)},
])

,Параметр,Значение
0,Корень проекта,c:\Users\user\Desktop\Практика\vk-vision-langu...
1,GQA-ru,c:\Users\user\Desktop\Практика\vk-vision-langu...
2,MMBench-ru,c:\Users\user\Desktop\Практика\vk-vision-langu...
3,CUDA,True
4,GPU,NVIDIA GeForce RTX 3080
5,dtype,torch.bfloat16
6,Checkpoint,c:\Users\user\Desktop\Практика\vk-vision-langu...


## 3. Предварительная проверка файлов и режима запуска

In [3]:
required_paths = [
    GQA_INSTRUCTIONS_PATH,
    GQA_IMAGES_PATH,
    MMBENCH_PATH,
    QWEN_MODEL_DIR,
    SMOL_MODEL_DIR,
]
required_paths.extend(spec["adapter_dir"] for spec in MODEL_SPECS if spec["adapter_dir"] is not None)

missing_paths = [path for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError("Не найдены обязательные файлы:\n" + "\n".join(map(str, missing_paths)))

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("REQUIRE_CUDA=True, но CUDA недоступна. Полный прогон на CPU заблокирован.")

run_config = {
    "version": 1,
    "max_gqa_samples": MAX_GQA_SAMPLES,
    "max_mmbench_samples": MAX_MMBENCH_SAMPLES,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "smol_longest_edge": SMOL_LONGEST_EDGE,
    "models": [
        {
            "slug": spec["slug"],
            "model_dir": str(spec["model_dir"]),
            "adapter_dir": str(spec["adapter_dir"]) if spec["adapter_dir"] else None,
        }
        for spec in MODEL_SPECS
    ],
}
run_config_json = json.dumps(run_config, ensure_ascii=False, sort_keys=True, indent=2)
run_hash = hashlib.sha256(run_config_json.encode("utf-8")).hexdigest()
run_metadata_path = CHECKPOINT_DIR / "run_config.json"

if run_metadata_path.exists():
    previous_config = json.loads(run_metadata_path.read_text(encoding="utf-8"))
    previous_hash = previous_config.get("run_hash")
    if previous_hash != run_hash:
        raise RuntimeError(
            "Конфигурация не совпадает с checkpoint-каталогом. "
            "Измените run_suffix/параметры или перенесите старый checkpoint-каталог."
        )
else:
    run_metadata_path.write_text(
        json.dumps({"run_hash": run_hash, "config": run_config}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("Предварительная проверка пройдена.")
print("Run hash:", run_hash)

Предварительная проверка пройдена.
Run hash: 8596a4b0b53ddad8cddc617fd58681a8b8a0ac4fc70f16562e4f1f3afb22454b


## 4. Загрузка всех тестовых данных

In [4]:
gqa = pd.read_parquet(GQA_INSTRUCTIONS_PATH).copy()
gqa["id"] = gqa["id"].astype(str)
gqa["imageId"] = gqa["imageId"].astype(str)
gqa["row_id"] = np.arange(len(gqa), dtype=int)

if MAX_GQA_SAMPLES is not None:
    gqa = gqa.sample(n=min(MAX_GQA_SAMPLES, len(gqa)), random_state=SEED).sort_values("row_id").reset_index(drop=True)

gqa_images = pd.read_parquet(GQA_IMAGES_PATH, columns=["id", "image"])
gqa_images["id"] = gqa_images["id"].astype(str)
gqa_image_map = dict(zip(gqa_images["id"], gqa_images["image"]))

missing_gqa_images = sorted(set(gqa["imageId"]) - set(gqa_image_map))
if missing_gqa_images:
    raise ValueError(f"Для {len(missing_gqa_images)} imageId нет изображения")

mmbench = pd.read_parquet(MMBENCH_PATH).copy()
mmbench["parquet_row_id"] = np.arange(len(mmbench), dtype=int)
mmbench["row_id"] = mmbench["parquet_row_id"]

if MAX_MMBENCH_SAMPLES is not None:
    mmbench = mmbench.sample(n=min(MAX_MMBENCH_SAMPLES, len(mmbench)), random_state=SEED).sort_values("parquet_row_id").reset_index(drop=True)

data_summary = pd.DataFrame([
    {"Датасет": "GQA-ru testdev", "Примеров": len(gqa), "Изображений": gqa["imageId"].nunique()},
    {"Датасет": "MMBench-ru dev", "Примеров": len(mmbench), "Изображений": len(mmbench)},
])
display(data_summary)
display(gqa[["row_id", "id", "imageId", "question", "answer"]].head(3))
display(mmbench[["row_id", "question", "answer", "category", "split"]].head(3))

,Датасет,Примеров,Изображений
0,GQA-ru testdev,12216,398
1,MMBench-ru dev,3910,3910


,row_id,id,imageId,question,answer
0,0,201307251,n161313,Пасмурно?,нет
1,1,201640614,n235859,Кто носит платье?,женщины
2,2,202225914,n336443,Посуда на столе выглядит чистой и черной?,нет


,row_id,question,answer,category,split
0,0,Какая часть яблони может вырасти в новое дерево?,A,physical_property_reasoning,dev
1,1,Из какого материала сделана эта лопатка?,A,function_reasoning,dev
2,2,Это место многолюдное?,A,image_scene,dev


## 5. Нормализация ответов и подготовка промптов

In [5]:
YES_NO_ALIASES = {
    "yes": "yes", "y": "yes", "yeah": "yes", "yep": "yes", "true": "yes",
    "да": "yes", "ага": "yes", "верно": "yes",
    "no": "no", "n": "no", "nope": "no", "false": "no",
    "нет": "no", "неа": "no",
}
OPTION_LABELS = ["A", "B", "C", "D"]
CYRILLIC_OPTION_MAP = str.maketrans({
    "А": "A", "В": "B", "С": "C", "Д": "D",
    "а": "A", "в": "B", "с": "C", "д": "D",
})


def is_missing_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return True
    return str(value).strip().lower() in {"", "nan", "none", "null", "<na>"}


def normalize_answer(text):
    if is_missing_text(text):
        return ""
    text = str(text).lower().strip().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    if text in YES_NO_ALIASES:
        return YES_NO_ALIASES[text]
    first_token = text.split()[0] if text else ""
    return YES_NO_ALIASES.get(first_token, text)


def gqa_prompt(question):
    return (
        "Answer the question using only the image. Give a short answer.\n"
        f"Question: {str(question).strip()}\nAnswer:"
    )


def get_options(row):
    return {
        label: str(row[label]).strip()
        for label in OPTION_LABELS
        if label in row.index and not is_missing_text(row[label])
    }


def options_to_text(options):
    return "\n".join(f"{label}. {text}" for label, text in options.items())


def normalize_choice_answer(answer, options):
    if is_missing_text(answer):
        return ""
    text = str(answer).strip().translate(CYRILLIC_OPTION_MAP).upper()
    if text in options:
        return text
    match = re.search(r"(?<![A-Z])([ABCD])(?![A-Z])", text)
    if match and match.group(1) in options:
        return match.group(1)
    answer_norm = normalize_answer(answer)
    for label, option_text in options.items():
        if answer_norm == normalize_answer(option_text):
            return label
    return text


def extract_choice_letter(prediction, options):
    if is_missing_text(prediction):
        return ""
    text = str(prediction).strip().translate(CYRILLIC_OPTION_MAP)
    text = re.sub(r"^\s*(assistant|ответ)\s*[:\-]\s*", "", text, flags=re.IGNORECASE)
    text_upper = text.upper()
    if text_upper[:1] in options:
        return text_upper[:1]
    match = re.search(r"(?<![A-Z])([ABCD])(?![A-Z])", text_upper)
    if match and match.group(1) in options:
        return match.group(1)
    prediction_norm = normalize_answer(text)
    for label, option_text in options.items():
        if prediction_norm == normalize_answer(option_text):
            return label
    return ""


def prepare_mmbench_example(row):
    options = get_options(row)
    hint = row.get("hint")
    question = str(row["question"]).strip()
    answer = row["answer"]
    if len(options) >= 2:
        prompt_parts = [
            "Ответь на вопрос по изображению.",
            "Выбери один правильный вариант из A, B, C или D.",
        ]
        if not is_missing_text(hint):
            prompt_parts.append(f"Подсказка: {str(hint).strip()}")
        prompt_parts.extend([
            f"Вопрос: {question}",
            "Варианты ответа:",
            options_to_text(options),
            "Ответь только одной буквой: A, B, C или D.",
        ])
        task_type = "multiple_choice"
        metric_name = "accuracy"
        true_answer_norm = normalize_choice_answer(answer, options)
    else:
        prompt_parts = ["Ответь на вопрос по изображению коротко: одним словом или короткой фразой."]
        if not is_missing_text(hint):
            prompt_parts.append(f"Подсказка: {str(hint).strip()}")
        prompt_parts.extend([f"Вопрос: {question}", "Ответ:"])
        task_type = "open_ended"
        metric_name = "exact_match"
        true_answer_norm = normalize_answer(answer)
    return {
        "row_id": int(row["row_id"]),
        "parquet_row_id": int(row["parquet_row_id"]),
        "split": str(row.get("split", "")),
        "category": str(row.get("category", "")),
        "question": question,
        "hint": "" if is_missing_text(hint) else str(hint).strip(),
        "options": options,
        "options_text": options_to_text(options),
        "true_answer": str(answer).strip(),
        "true_answer_norm": true_answer_norm,
        "task_type": task_type,
        "metric_name": metric_name,
        "prompt": "\n".join(prompt_parts),
    }

## 6. Работа с изображениями

In [6]:
def load_image_safely(image_value):
    try:
        if isinstance(image_value, Image.Image):
            return image_value.convert("RGB")
        if isinstance(image_value, dict):
            if image_value.get("bytes") is not None:
                return Image.open(BytesIO(image_value["bytes"])).convert("RGB")
            if image_value.get("path"):
                image_path = Path(image_value["path"])
                if not image_path.is_absolute():
                    image_path = ROOT / image_path
                return Image.open(image_path).convert("RGB")
        if isinstance(image_value, (bytes, bytearray)):
            return Image.open(BytesIO(image_value)).convert("RGB")
        if isinstance(image_value, (str, Path)):
            return Image.open(image_value).convert("RGB")
        raise TypeError(f"Неподдерживаемый тип изображения: {type(image_value)}")
    except (FileNotFoundError, OSError, UnidentifiedImageError, TypeError, ValueError) as error:
        raise ValueError(f"Не удалось открыть изображение: {error}") from error


# Быстрая проверка по одному изображению каждого датасета.
gqa_check = load_image_safely(gqa_image_map[gqa.iloc[0]["imageId"]])
mmbench_check = load_image_safely(mmbench.iloc[0]["image"])
print("GQA image:", gqa_check.size, gqa_check.mode)
print("MMBench image:", mmbench_check.size, mmbench_check.mode)
gqa_check.close()
mmbench_check.close()

GQA image: (500, 280) RGB
MMBench image: (512, 398) RGB


## 7. Загрузка моделей и генерация ответов

In [7]:
def get_model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type != "meta":
            return parameter.device
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def from_pretrained_with_dtype(model_class, source, dtype, **kwargs):
    try:
        return model_class.from_pretrained(source, dtype=dtype, **kwargs)
    except TypeError:
        return model_class.from_pretrained(source, torch_dtype=dtype, **kwargs)


def load_processor(spec):
    if spec["family"] == "qwen":
        kwargs = {
            "local_files_only": True,
            "min_pixels": MIN_PIXELS,
            "max_pixels": MAX_PIXELS,
        }
    else:
        kwargs = {
            "local_files_only": True,
            "size": {"longest_edge": SMOL_LONGEST_EDGE},
        }
    try:
        processor = AutoProcessor.from_pretrained(str(spec["model_dir"]), **kwargs)
    except TypeError:
        kwargs.pop("min_pixels", None)
        kwargs.pop("max_pixels", None)
        kwargs.pop("size", None)
        processor = AutoProcessor.from_pretrained(str(spec["model_dir"]), **kwargs)
    if getattr(processor, "tokenizer", None) is not None:
        if processor.tokenizer.pad_token is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token
        processor.tokenizer.padding_side = "left"
    return processor


def load_model_bundle(spec):
    load_started = time.perf_counter()
    processor = load_processor(spec)
    if spec["family"] == "qwen":
        model = from_pretrained_with_dtype(
            Qwen2_5_VLForConditionalGeneration,
            str(spec["model_dir"]),
            MODEL_DTYPE,
            local_files_only=True,
            device_map="auto" if torch.cuda.is_available() else None,
            _attn_implementation="eager",
        )
    else:
        model = from_pretrained_with_dtype(
            SmolVLMModelClass,
            str(spec["model_dir"]),
            MODEL_DTYPE,
            local_files_only=True,
            _attn_implementation="eager",
        )
        model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    if spec["adapter_dir"] is not None:
        model = PeftModel.from_pretrained(
            model,
            str(spec["adapter_dir"]),
            local_files_only=True,
            is_trainable=False,
        )
    model.eval()
    return {
        "spec": spec,
        "processor": processor,
        "model": model,
        "attention": "eager",
        "load_time_s": time.perf_counter() - load_started,
    }


def generate_qwen(bundle, image, prompt):
    processor = bundle["processor"]
    model = bundle["model"]
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(get_model_input_device(model))
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = [output[len(input_ids):] for input_ids, output in zip(inputs.input_ids, generated)]
    return processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


def generate_smol(bundle, image, prompt):
    processor = bundle["processor"]
    model = bundle["model"]
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, images=[image], return_tensors="pt").to(get_model_input_device(model))
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = generated[:, inputs.input_ids.shape[1]:]
    answer = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return re.sub(r"^\s*Assistant:\s*", "", answer).strip()


def predict(bundle, image, prompt):
    if bundle["spec"]["family"] == "qwen":
        return generate_qwen(bundle, image, prompt)
    return generate_smol(bundle, image, prompt)


def release_bundle(bundle):
    if bundle is not None:
        bundle.pop("model", None)
        bundle.pop("processor", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 8. Промежуточные файлы и возобновление запуска

In [8]:
def checkpoint_path(spec, dataset_name):
    return CHECKPOINT_DIR / f"{spec['slug']}__{dataset_name}.csv"


def runtime_info_path(spec):
    return CHECKPOINT_DIR / f"{spec['slug']}__runtime.json"


def atomic_to_csv(dataframe, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    dataframe.to_csv(temporary_path, index=False, encoding="utf-8-sig")
    temporary_path.replace(path)


def read_checkpoint(spec, dataset_name):
    path = checkpoint_path(spec, dataset_name)
    if not path.exists():
        return pd.DataFrame()
    checkpoint = pd.read_csv(path, dtype={"id": str, "image_id": str})
    if "row_id" in checkpoint:
        checkpoint = checkpoint.sort_values("row_id").drop_duplicates("row_id", keep="last")
    return checkpoint.reset_index(drop=True)


def checkpoint_is_complete(spec, dataset_name, expected_row_ids):
    checkpoint = read_checkpoint(spec, dataset_name)
    return (
        len(checkpoint) == len(expected_row_ids)
        and set(checkpoint["row_id"].astype(int)) == set(map(int, expected_row_ids))
        and (checkpoint["error"].isna().all() or not STRICT_FULL_COVERAGE)
    )


def append_and_save(records, existing, path):
    new_rows = pd.DataFrame(records)
    combined = pd.concat([existing, new_rows], ignore_index=True) if not existing.empty else new_rows
    combined = combined.sort_values("row_id").drop_duplicates("row_id", keep="last").reset_index(drop=True)
    atomic_to_csv(combined, path)
    return combined

## 9. Полный инференс с промежуточным сохранением

In [9]:
def evaluate_gqa(spec, bundle):
    path = checkpoint_path(spec, "gqa")
    existing = read_checkpoint(spec, "gqa")
    completed = set(existing.loc[existing["error"].isna(), "row_id"].astype(int)) if not existing.empty else set()
    pending = gqa[~gqa["row_id"].isin(completed)]
    records = []

    for _, row in tqdm(pending.iterrows(), total=len(pending), desc=f"GQA | {spec['name']}"):
        image = None
        prediction = ""
        error = None
        started = time.perf_counter()
        try:
            image = load_image_safely(gqa_image_map[str(row["imageId"])])
            prediction = predict(bundle, image, gqa_prompt(row["question"]))
        except Exception as exc:
            error = str(exc)
            if isinstance(exc, RuntimeError) and torch.cuda.is_available():
                torch.cuda.empty_cache()
        finally:
            inference_time_s = time.perf_counter() - started
            if image is not None:
                image.close()

        prediction_norm = normalize_answer(prediction)
        true_norm = normalize_answer(row["answer"])
        records.append({
            "row_id": int(row["row_id"]),
            "id": str(row["id"]),
            "image_id": str(row["imageId"]),
            "question": str(row["question"]),
            "true_answer": str(row["answer"]),
            "true_answer_norm": true_norm,
            "prediction": prediction,
            "prediction_norm": prediction_norm,
            "is_correct": bool(error is None and prediction_norm == true_norm),
            "inference_time_s": inference_time_s,
            "error": error,
        })
        if len(records) >= CHECKPOINT_EVERY:
            existing = append_and_save(records, existing, path)
            records = []

    if records:
        existing = append_and_save(records, existing, path)
    return existing.sort_values("row_id").reset_index(drop=True)


def evaluate_mmbench(spec, bundle):
    path = checkpoint_path(spec, "mmbench")
    existing = read_checkpoint(spec, "mmbench")
    completed = set(existing.loc[existing["error"].isna(), "row_id"].astype(int)) if not existing.empty else set()
    pending = mmbench[~mmbench["row_id"].isin(completed)]
    records = []

    for _, row in tqdm(pending.iterrows(), total=len(pending), desc=f"MMBench | {spec['name']}"):
        example = prepare_mmbench_example(row)
        image = None
        prediction = ""
        error = None
        started = time.perf_counter()
        try:
            image = load_image_safely(row["image"])
            prediction = predict(bundle, image, example["prompt"])
            if example["task_type"] == "multiple_choice":
                prediction_norm = extract_choice_letter(prediction, example["options"])
            else:
                prediction_norm = normalize_answer(prediction)
        except Exception as exc:
            error = str(exc)
            prediction_norm = ""
            if isinstance(exc, RuntimeError) and torch.cuda.is_available():
                torch.cuda.empty_cache()
        finally:
            inference_time_s = time.perf_counter() - started
            if image is not None:
                image.close()

        records.append({
            "model_name": spec["name"],
            "family": spec["family"],
            "row_id": example["row_id"],
            "parquet_row_id": example["parquet_row_id"],
            "split": example["split"],
            "category": example["category"],
            "task_type": example["task_type"],
            "metric_name": example["metric_name"],
            "question": example["question"],
            "hint": example["hint"],
            "options": example["options_text"],
            "true_answer": example["true_answer"],
            "true_answer_norm": example["true_answer_norm"],
            "prediction": prediction,
            "prediction_norm": prediction_norm,
            "is_correct": bool(error is None and prediction_norm == example["true_answer_norm"]),
            "inference_time_s": inference_time_s,
            "error": error,
        })
        if len(records) >= CHECKPOINT_EVERY:
            existing = append_and_save(records, existing, path)
            records = []

    if records:
        existing = append_and_save(records, existing, path)
    return existing.sort_values("row_id").reset_index(drop=True)

## 10. Запуск всех моделей

Ячейка ниже — самая длительная. Модель загружается только тогда, когда для неё не завершён хотя бы один датасет. После GQA-ru та же загруженная модель сразу оценивается на MMBench-ru, затем память освобождается.

In [10]:
gqa_expected_ids = gqa["row_id"].astype(int).tolist()
mmbench_expected_ids = mmbench["row_id"].astype(int).tolist()

for spec in MODEL_SPECS:
    gqa_done = checkpoint_is_complete(spec, "gqa", gqa_expected_ids)
    mmbench_done = checkpoint_is_complete(spec, "mmbench", mmbench_expected_ids)
    print(f"\n{spec['name']}: GQA={'готово' if gqa_done else 'нужно'}, MMBench={'готово' if mmbench_done else 'нужно'}")
    if gqa_done and mmbench_done:
        continue

    bundle = None
    try:
        bundle = load_model_bundle(spec)
        runtime_info_path(spec).write_text(
            json.dumps({
                "model_name": spec["name"],
                "attention": bundle["attention"],
                "load_time_s": bundle["load_time_s"],
            }, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        if not gqa_done:
            evaluate_gqa(spec, bundle)
        if not mmbench_done:
            evaluate_mmbench(spec, bundle)
    finally:
        release_bundle(bundle)

print("Инференс всех доступных частей завершён.")


Qwen2.5-VL базовая: GQA=нужно, MMBench=нужно


Loading weights: 100%|██████████| 824/824 [00:02<00:00, 365.39it/s]
GQA | Qwen2.5-VL базовая: 100%|██████████| 12216/12216 [1:01:39<00:00,  3.30it/s]
MMBench | Qwen2.5-VL базовая: 100%|██████████| 3910/3910 [16:01<00:00,  4.07it/s]



Qwen2.5-VL + LoRA #1: GQA=нужно, MMBench=нужно


Loading weights: 100%|██████████| 824/824 [00:02<00:00, 390.96it/s]
GQA | Qwen2.5-VL + LoRA #1: 100%|██████████| 12216/12216 [1:31:59<00:00,  2.21it/s]
MMBench | Qwen2.5-VL + LoRA #1: 100%|██████████| 3910/3910 [21:01<00:00,  3.10it/s]



Qwen2.5-VL + LoRA #2: GQA=нужно, MMBench=нужно


...


SmolVLM базовая: GQA=нужно, MMBench=нужно


Loading weights: 100%|██████████| 657/657 [00:00<00:00, 3432.35it/s]
GQA | SmolVLM базовая: 100%|██████████| 12216/12216 [1:00:07<00:00,  3.39it/s]
MMBench | SmolVLM базовая: 100%|██████████| 3910/3910 [18:27<00:00,  3.53it/s]



SmolVLM + LoRA: GQA=нужно, MMBench=нужно


Loading weights: 100%|██████████| 657/657 [00:00<00:00, 11731.69it/s]
GQA | SmolVLM + LoRA: 100%|██████████| 12216/12216 [1:32:06<00:00,  2.21it/s]
MMBench | SmolVLM + LoRA: 100%|██████████| 3910/3910 [22:07<00:00,  2.95it/s]

Инференс всех доступных частей завершён.


## 11. Проверка полного покрытия

In [11]:
coverage_rows = []
coverage_failures = []

for spec in MODEL_SPECS:
    for dataset_name, expected_ids in [
        ("gqa", gqa_expected_ids),
        ("mmbench", mmbench_expected_ids),
    ]:
        part = read_checkpoint(spec, dataset_name)
        expected_count = len(expected_ids)
        actual_count = len(part)
        errors = int(part["error"].notna().sum()) if "error" in part else actual_count
        ids_match = set(part["row_id"].astype(int)) == set(expected_ids) if actual_count else False
        valid = actual_count == expected_count and ids_match and (errors == 0 or not STRICT_FULL_COVERAGE)
        coverage_rows.append({
            "Модель": spec["name"],
            "Датасет": dataset_name,
            "Ожидалось": expected_count,
            "Получено": actual_count,
            "Ошибок": errors,
            "Полное покрытие": valid,
        })
        if not valid:
            coverage_failures.append((spec["slug"], dataset_name, actual_count, errors))

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

is_full_mode = MAX_GQA_SAMPLES is None and MAX_MMBENCH_SAMPLES is None
if not is_full_mode:
    raise RuntimeError(
        "Запуск выполнен с ограничением числа примеров. Основные CSV не будут перезаписаны. "
        "Верните MAX_GQA_SAMPLES=None и MAX_MMBENCH_SAMPLES=None."
    )
if coverage_failures:
    raise RuntimeError(f"Публикация остановлена: неполное покрытие или ошибки: {coverage_failures}")

,Модель,Датасет,Ожидалось,Получено,Ошибок,Полное покрытие
0,Qwen2.5-VL базовая,gqa,12216,12216,0,True
1,Qwen2.5-VL базовая,mmbench,3910,3910,0,True
2,Qwen2.5-VL + LoRA #1,gqa,12216,12216,0,True
3,Qwen2.5-VL + LoRA #1,mmbench,3910,3910,0,True
4,Qwen2.5-VL + LoRA #2,gqa,12216,12216,0,True
5,Qwen2.5-VL + LoRA #2,mmbench,3910,3910,0,True
6,Qwen2.5-VL + LoRA #3,gqa,12216,12216,0,True
7,Qwen2.5-VL + LoRA #3,mmbench,3910,3910,0,True
8,SmolVLM базовая,gqa,12216,12216,0,True
9,SmolVLM базовая,mmbench,3910,3910,0,True


## 12. Формирование совместимых GQA-ru CSV

In [12]:
def classify_pair(row):
    if row["lora_correct"] and not row["baseline_correct"]:
        return "LoRA лучше"
    if row["baseline_correct"] and not row["lora_correct"]:
        return "Baseline лучше"
    if row["baseline_correct"] and row["lora_correct"]:
        return "Обе верно"
    return "Обе неверно"


def build_gqa_pair_file(base_slug, adapter_slug):
    base_spec = next(spec for spec in MODEL_SPECS if spec["slug"] == base_slug)
    adapter_spec = next(spec for spec in MODEL_SPECS if spec["slug"] == adapter_slug)
    base = read_checkpoint(base_spec, "gqa")
    adapter = read_checkpoint(adapter_spec, "gqa")

    base = base.rename(columns={
        "prediction": "baseline_answer",
        "prediction_norm": "baseline_answer_norm",
        "is_correct": "baseline_correct",
        "error": "baseline_answer_error",
    })
    adapter = adapter[["row_id", "prediction", "prediction_norm", "is_correct", "error"]].rename(columns={
        "prediction": "lora_answer",
        "prediction_norm": "lora_answer_norm",
        "is_correct": "lora_correct",
        "error": "lora_answer_error",
    })
    pair = base.merge(adapter, on="row_id", how="inner", validate="one_to_one")
    pair["image_path"] = None
    pair["result_type"] = pair.apply(classify_pair, axis=1)
    columns = [
        "row_id", "id", "image_id", "image_path", "question", "true_answer",
        "baseline_answer", "lora_answer", "true_answer_norm",
        "baseline_answer_norm", "lora_answer_norm", "baseline_correct",
        "lora_correct", "result_type", "baseline_answer_error", "lora_answer_error",
    ]
    return pair[columns].sort_values("row_id").reset_index(drop=True)


def build_gqa_metrics(pair, family):
    baseline_name = "Baseline Qwen2.5-VL" if family == "qwen" else "Baseline SmolVLM"
    lora_name = "Qwen2.5-VL + LoRA" if family == "qwen" else "SmolVLM + LoRA"
    return pd.DataFrame([
        {
            "Модель": baseline_name,
            "Accuracy / Exact Match": float(pair["baseline_correct"].mean()),
            "Количество верных ответов": int(pair["baseline_correct"].sum()),
            "Количество примеров": len(pair),
        },
        {
            "Модель": lora_name,
            "Accuracy / Exact Match": float(pair["lora_correct"].mean()),
            "Количество верных ответов": int(pair["lora_correct"].sum()),
            "Количество примеров": len(pair),
        },
    ])


gqa_publication = {}
for adapter_slug, (predictions_name, metrics_name, base_slug) in GQA_EXPORTS.items():
    pair = build_gqa_pair_file(base_slug, adapter_slug)
    family = next(spec["family"] for spec in MODEL_SPECS if spec["slug"] == adapter_slug)
    gqa_publication[predictions_name] = pair
    gqa_publication[metrics_name] = build_gqa_metrics(pair, family)

display(pd.concat([
    frame.assign(Файл=name)
    for name, frame in gqa_publication.items()
    if name.endswith("_metrics.csv")
], ignore_index=True))

,Модель,Accuracy / Exact Match,Количество верных ответов,Количество примеров,Файл
0,Baseline Qwen2.5-VL,0.319990,3909,12216,qwen_lora_metrics.csv
1,Qwen2.5-VL + LoRA,0.500655,6116,12216,qwen_lora_metrics.csv
2,Baseline Qwen2.5-VL,0.319990,3909,12216,qwen_lora_v2_metrics.csv
3,Qwen2.5-VL + LoRA,0.537246,6563,12216,qwen_lora_v2_metrics.csv
4,Baseline Qwen2.5-VL,0.319990,3909,12216,qwen_lora_v3_metrics.csv
5,Qwen2.5-VL + LoRA,0.535282,6539,12216,qwen_lora_v3_metrics.csv
6,Baseline SmolVLM,0.202849,2478,12216,smolvlm_lora_metrics.csv
7,SmolVLM + LoRA,0.291830,3565,12216,smolvlm_lora_metrics.csv


## 13. Формирование совместимых MMBench-ru CSV

In [13]:
mmbench_parts = []
mmbench_metric_rows = []

for spec in MODEL_SPECS:
    part = read_checkpoint(spec, "mmbench")
    part["model_name"] = spec["name"]
    part["family"] = spec["family"]
    mmbench_parts.append(part)

    successful = part[part["error"].isna()].copy()
    multiple_choice = successful[successful["task_type"] == "multiple_choice"]
    open_ended = successful[successful["task_type"] == "open_ended"]
    runtime_path = runtime_info_path(spec)
    runtime_info = json.loads(runtime_path.read_text(encoding="utf-8")) if runtime_path.exists() else {}
    mmbench_metric_rows.append({
        "model_name": spec["name"],
        "family": spec["family"],
        "status": "ok",
        "main_metric": "accuracy" if len(open_ended) == 0 else "accuracy / exact_match",
        "main_score": float(successful["is_correct"].mean()),
        "accuracy": float(multiple_choice["is_correct"].mean()) if len(multiple_choice) else np.nan,
        "exact_match": float(open_ended["is_correct"].mean()) if len(open_ended) else np.nan,
        "num_success": len(successful),
        "num_errors": int(part["error"].notna().sum()),
        "num_correct": int(successful["is_correct"].sum()),
        "mean_inference_time_s": float(successful["inference_time_s"].mean()),
        "attention": runtime_info.get("attention", "eager"),
        "load_error": None,
        "load_time_s": runtime_info.get("load_time_s", np.nan),
    })

mmbench_predictions = pd.concat(mmbench_parts, ignore_index=True)
mmbench_metrics = pd.DataFrame(mmbench_metric_rows)

mmbench_columns = [
    "model_name", "family", "row_id", "parquet_row_id", "split", "category",
    "task_type", "metric_name", "question", "hint", "options", "true_answer",
    "true_answer_norm", "prediction", "prediction_norm", "is_correct",
    "inference_time_s", "error",
]
mmbench_predictions = mmbench_predictions[mmbench_columns]

display(mmbench_metrics)

,model_name,family,status,main_metric,main_score,accuracy,exact_match,num_success,num_errors,num_correct,mean_inference_time_s,attention,load_error,load_time_s
0,Qwen2.5-VL базовая,qwen,ok,accuracy,0.799233,0.799233,NaN,3910,0,3125,0.244883,eager,None,3.560898
1,Qwen2.5-VL + LoRA #1,qwen,ok,accuracy,0.797954,0.797954,NaN,3910,0,3120,0.321671,eager,None,4.757903
2,Qwen2.5-VL + LoRA #2,qwen,ok,accuracy,0.803325,0.803325,NaN,3910,0,3141,0.323854,eager,None,4.183942
3,Qwen2.5-VL + LoRA #3,qwen,ok,accuracy,0.806138,0.806138,NaN,3910,0,3152,0.336779,eager,None,4.512859
4,SmolVLM базовая,smolvlm,ok,accuracy,0.373402,0.373402,NaN,3910,0,1460,0.282201,eager,None,6.467476
5,SmolVLM + LoRA,smolvlm,ok,accuracy,0.365473,0.365473,NaN,3910,0,1429,0.338445,eager,None,2.380438


## 14. Резервная копия и атомарная публикация

In [14]:
publication = dict(gqa_publication)
publication["mmbench_ru_predictions.csv"] = mmbench_predictions
publication["mmbench_ru_metrics.csv"] = mmbench_metrics

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = REPORT_DIR / "backups" / f"before_full_eval_{timestamp}"
backup_dir.mkdir(parents=True, exist_ok=True)

for file_name in publication:
    current_path = REPORT_DIR / file_name
    if current_path.exists():
        shutil.copy2(current_path, backup_dir / file_name)

for file_name, dataframe in publication.items():
    atomic_to_csv(dataframe, REPORT_DIR / file_name)

publish_manifest = pd.DataFrame([
    {
        "Файл": file_name,
        "Строк": len(dataframe),
        "Путь": str(REPORT_DIR / file_name),
    }
    for file_name, dataframe in publication.items()
])
display(publish_manifest)
print("Резервная копия прежних CSV:", backup_dir)

,Файл,Строк,Путь
0,qwen_lora_predictions.csv,12216,c:\Users\user\Desktop\Практика\vk-vision-langu...
1,qwen_lora_metrics.csv,2,c:\Users\user\Desktop\Практика\vk-vision-langu...
2,qwen_lora_v2_predictions.csv,12216,c:\Users\user\Desktop\Практика\vk-vision-langu...
3,qwen_lora_v2_metrics.csv,2,c:\Users\user\Desktop\Практика\vk-vision-langu...
4,qwen_lora_v3_predictions.csv,12216,c:\Users\user\Desktop\Практика\vk-vision-langu...
5,qwen_lora_v3_metrics.csv,2,c:\Users\user\Desktop\Практика\vk-vision-langu...
6,smolvlm_lora_predictions.csv,12216,c:\Users\user\Desktop\Практика\vk-vision-langu...
7,smolvlm_lora_metrics.csv,2,c:\Users\user\Desktop\Практика\vk-vision-langu...
8,mmbench_ru_predictions.csv,23460,c:\Users\user\Desktop\Практика\vk-vision-langu...
9,mmbench_ru_metrics.csv,6,c:\Users\user\Desktop\Практика\vk-vision-langu...


Резервная копия прежних CSV: c:\Users\user\Desktop\Практика\vk-vision-language-vqa\reports\evaluation\backups\before_full_eval_20260713_090942


## 15. Проверка опубликованных файлов

In [15]:
verification_rows = []

for file_name, expected in publication.items():
    published = pd.read_csv(REPORT_DIR / file_name)
    columns_match = list(published.columns) == list(expected.columns)
    rows_match = len(published) == len(expected)
    verification_rows.append({
        "Файл": file_name,
        "Ожидалось строк": len(expected),
        "Опубликовано строк": len(published),
        "Колонки совпадают": columns_match,
        "Проверка пройдена": columns_match and rows_match,
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)
if not verification_df["Проверка пройдена"].all():
    raise RuntimeError("Проверка опубликованных CSV не пройдена")

print("Все итоговые файлы готовы для notebooks/evaluate_all_models.ipynb")

,Файл,Ожидалось строк,Опубликовано строк,Колонки совпадают,Проверка пройдена
0,qwen_lora_predictions.csv,12216,12216,True,True
1,qwen_lora_metrics.csv,2,2,True,True
2,qwen_lora_v2_predictions.csv,12216,12216,True,True
3,qwen_lora_v2_metrics.csv,2,2,True,True
4,qwen_lora_v3_predictions.csv,12216,12216,True,True
5,qwen_lora_v3_metrics.csv,2,2,True,True
6,smolvlm_lora_predictions.csv,12216,12216,True,True
7,smolvlm_lora_metrics.csv,2,2,True,True
8,mmbench_ru_predictions.csv,23460,23460,True,True
9,mmbench_ru_metrics.csv,6,6,True,True


Все итоговые файлы готовы для notebooks/evaluate_all_models.ipynb
